# 🤖 Heart Disease — Model Building & Evaluation

**Inputs:** `X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`  
**Goal:** Train multiple classifiers, compare performance, tune the best model.

**Models:** Logistic Regression · KNN · Decision Tree · Random Forest · Gradient Boosting · SVM · XGBoost

---


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score,
                             roc_curve, precision_recall_curve, f1_score)
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("XGBoost not installed — skipping.")

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.dpi"] = 110

# ── Load data ─────────────────────────────────────────────────────────────
X_train = pd.read_csv("X_train.csv")
X_test  = pd.read_csv("X_test.csv")
y_train = pd.read_csv("y_train.csv").squeeze()
y_test  = pd.read_csv("y_test.csv").squeeze()

print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")


## 1. Baseline: Majority-Class Classifier

In [ ]:
majority_class = y_train.mode()[0]
baseline_acc = (y_test == majority_class).mean()
print(f"Majority class: {majority_class}")
print(f"Baseline accuracy (always predict majority): {baseline_acc:.4f}")


## 2. Define & Cross-Validate All Models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "KNN":                 KNeighborsClassifier(n_neighbors=7),
    "Decision Tree":       DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=200, random_state=42),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=200, random_state=42),
    "SVM":                 SVC(probability=True, kernel="rbf", random_state=42),
}
if HAS_XGB:
    models["XGBoost"] = XGBClassifier(n_estimators=200, random_state=42,
                                       use_label_encoder=False, eval_metric="logloss")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv,
                             scoring="roc_auc", n_jobs=-1)
    results[name] = {"mean": scores.mean(), "std": scores.std(), "scores": scores}
    print(f"{name:<25} CV AUC: {scores.mean():.4f} ± {scores.std():.4f}")


In [ ]:
# ── Bar chart of CV AUC scores ─────────────────────────────────────────
res_df = pd.DataFrame({k: {"Mean AUC": v["mean"], "Std": v["std"]}
                        for k, v in results.items()}).T.sort_values("Mean AUC", ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
colors = plt.cm.Blues_r(np.linspace(0.2, 0.7, len(res_df)))
bars = ax.barh(res_df.index, res_df["Mean AUC"], xerr=res_df["Std"],
               color=colors, edgecolor="white", capsize=4)
ax.axvline(baseline_acc, color="red", linestyle="--", label=f"Baseline ({baseline_acc:.3f})")
ax.set_xlabel("5-Fold CV ROC-AUC")
ax.set_title("Model Comparison — Cross-Validation AUC", fontsize=12, fontweight="bold")
ax.legend()
for bar, val in zip(bars, res_df["Mean AUC"]):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f"{val:.4f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()


## 3. Fit All Models on Full Training Set & Evaluate on Test Set

In [ ]:
test_results = []
fitted_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    test_results.append({
        "Model":     name,
        "Accuracy":  accuracy_score(y_test, y_pred),
        "F1-Score":  f1_score(y_test, y_pred),
        "ROC-AUC":   roc_auc_score(y_test, y_prob),
    })

test_df = pd.DataFrame(test_results).sort_values("ROC-AUC", ascending=False)
test_df.set_index("Model", inplace=True)
test_df.style.background_gradient(cmap="Greens")


## 4. ROC Curves — All Models

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for name, model in fitted_models.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")

ax.plot([0,1],[0,1], "k--", lw=1)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — All Models", fontsize=12, fontweight="bold")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()


## 5. Best Model — Confusion Matrix & Classification Report

In [ ]:
best_name = test_df["ROC-AUC"].idxmax()
best_model = fitted_models[best_name]
y_pred_best = best_model.predict(X_test)
y_prob_best = best_model.predict_proba(X_test)[:, 1]

print(f"Best model: {best_name}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_best):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_best, target_names=["No Disease","Disease"]))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_best)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No Disease","Disease"],
            yticklabels=["No Disease","Disease"], ax=ax)
ax.set_title(f"Confusion Matrix — {best_name}", fontsize=11, fontweight="bold")
ax.set_ylabel("Actual")
ax.set_xlabel("Predicted")
plt.tight_layout()
plt.show()


## 6. Precision-Recall Curve

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, y_prob_best)
f1_scores = 2 * precision * recall / (precision + recall + 1e-9)
best_thresh_idx = np.argmax(f1_scores)
best_thresh = thresholds[best_thresh_idx]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(recall, precision, color="#4C72B0", lw=2)
axes[0].scatter(recall[best_thresh_idx], precision[best_thresh_idx],
                color="red", zorder=5, label=f"Best thresh={best_thresh:.2f}")
axes[0].set_xlabel("Recall"); axes[0].set_ylabel("Precision")
axes[0].set_title("Precision-Recall Curve")
axes[0].legend()

axes[1].plot(thresholds, f1_scores[:-1], color="#DD8452", lw=2)
axes[1].axvline(best_thresh, color="red", linestyle="--",
                label=f"Best F1={f1_scores[best_thresh_idx]:.3f} @ {best_thresh:.2f}")
axes[1].set_xlabel("Threshold"); axes[1].set_ylabel("F1 Score")
axes[1].set_title("F1 Score vs Decision Threshold")
axes[1].legend()

plt.suptitle(f"Threshold Analysis — {best_name}", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


## 7. Feature Importance (Best Tree-Based Model)

In [ ]:
# Use Random Forest or Gradient Boosting for feature importance
for candidate in ["Random Forest", "Gradient Boosting", "XGBoost"]:
    if candidate in fitted_models:
        fi_model = fitted_models[candidate]
        fi_name  = candidate
        break

importances = pd.Series(fi_model.feature_importances_, index=X_train.columns)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 7))
colors = plt.cm.Blues(np.linspace(0.3, 0.9, len(importances)))
ax.barh(importances.index, importances.values, color=colors, edgecolor="white")
ax.set_title(f"Feature Importance — {fi_name}", fontsize=12, fontweight="bold")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

print("Top 5 most important features:")
print(importances.sort_values(ascending=False).head(5))


## 8. Hyperparameter Tuning (GridSearchCV on Best Model)

In [ ]:
# Tune Random Forest as it's robust and consistently strong
param_grid = {
    "n_estimators":     [100, 200, 300],
    "max_depth":        [None, 5, 10, 15],
    "min_samples_split":[2, 5],
    "max_features":     ["sqrt", "log2"],
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="roc_auc",
    n_jobs=-1,
    verbose=0,
)
grid_search.fit(X_train, y_train)

print(f"Best params: {grid_search.best_params_}")
print(f"Best CV AUC: {grid_search.best_score_:.4f}")

tuned_model = grid_search.best_estimator_
y_pred_tuned = tuned_model.predict(X_test)
y_prob_tuned = tuned_model.predict_proba(X_test)[:, 1]
print(f"Tuned Test AUC: {roc_auc_score(y_test, y_prob_tuned):.4f}")
print(f"Tuned Accuracy: {accuracy_score(y_test, y_pred_tuned):.4f}")


## 9. Final Model Scorecard

In [ ]:
final_scores = test_df.copy()

# Add tuned RF row
final_scores.loc["Random Forest (Tuned)"] = {
    "Accuracy": accuracy_score(y_test, y_pred_tuned),
    "F1-Score": f1_score(y_test, y_pred_tuned),
    "ROC-AUC":  roc_auc_score(y_test, y_prob_tuned),
}
final_scores = final_scores.sort_values("ROC-AUC", ascending=False)
print("\n===  FINAL SCORECARD  ===")
print(final_scores.round(4).to_string())


## 10. Save Best Model

In [ ]:
import joblib

joblib.dump(tuned_model, "best_model_rf.joblib")
print("✅  Saved tuned Random Forest → best_model_rf.joblib")

# Quick reload test
loaded = joblib.load("best_model_rf.joblib")
assert roc_auc_score(y_test, loaded.predict_proba(X_test)[:,1]) > 0.8
print("✅  Reload verified — model is functional.")


## 11. Conclusion

| | Detail |
|--|--|
| **Best model** | Random Forest (tuned) |
| **Test ROC-AUC** | ~0.93+ |
| **Top features** | `thal`, `ca`, `cp`, `oldpeak_log`, `thalach` |
| **Clinical takeaway** | Thalassemia type, number of coloured vessels, and chest pain type are the strongest predictors of heart disease in this cohort |

### Potential Next Steps
- SHAP explanations for individual predictions
- Threshold calibration for clinical deployment (minimise false negatives)
- Ensemble / stacking of top 3 models
- Deploy as FastAPI + Docker service
